# SOTA Championship Cross-Modal Person Retrieval: Kaggle Dual Tesla T4
### WACV 2027 UPAR Challenge & MEVID Benchmark | Target: $\ge 0.9000$ mADM

This notebook executes the unconstrained championship pipeline directly from the cloned repository (`sota_pipeline`).

**Key Capabilities:**
- **Rank 1 HDF5 Binary Container Support**: Connects to `upar_binary_dataset.h5` in <2 seconds, eliminating Kaggle network FUSE inode lag.
- **Lightning Fast RAM Preloading**: Reads 97,669 images into host RAM in ~2.5 seconds (only ~650 MB RAM used).
- **Distributed DDP Multi-GPU**: Dual Tesla T4 (2x15GB VRAM) with Automatic Mixed Precision (AMP FP16).
- **Max VRAM Utilization**: Batch size 64/GPU (128 global, 256 effective with 2 accumulation steps) utilizing ~13.5 GB VRAM/GPU.
- **Informative Step-Wise Logging**: Clean formatted metrics every 25 steps (NO tqdm bar clutter) with loss components, current epoch ETA, total run ETA, img/s throughput, and per-GPU VRAM telemetry.
- **Baseline Benchmark (Epoch 0)**: Evaluates zero-shot performance prior to training.
- **Full Per-Epoch Validation**: Computes continuous 1024-D metric distance, attribute surprisal L1, Reciprocal Rank Fusion (RRF), mADM, CMC (Rank-1, 5, 10), and mAP after every single epoch.
- **Strict Checkpoint Hygiene**: Saves ONLY `best_model.pt` and `last_checkpoint.pt` in raw uncompressed format with all 7 components.
- **Post-Training Integrity Audit**: Verifies checkpoints, weights, and model loading.


In [ ]:
# ==============================================================================
# CELL 1: HARDWARE & SYSTEM TELEMETRY CHECK
# ==============================================================================
import os
import sys
import psutil
import torch

print("=" * 80)
print("[HARDWARE] Inspecting Dual Tesla T4 Accelerators & Host Environment...")
print("=" * 80)

cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f"CUDA Available: {cuda_available} | Device Count: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  -> GPU {i}: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")

vmem = psutil.virtual_memory()
print(f"Host CPU Cores: {os.cpu_count()} | Total RAM: {vmem.total / (1024**3):.2f} GB | Available: {vmem.available / (1024**3):.2f} GB")
print("=" * 80)

# Install required SOTA vision-language dependencies
print("[DEPS] Installing OpenCLIP and Timm dependencies...")
import subprocess
subprocess.run(["pip", "install", "-q", "open-clip-torch", "timm"], check=True)
print("[DEPS] Dependencies installed successfully!")


In [ ]:
# ==============================================================================
# CELL 2: CLONE OR PULL PRIVATE REPOSITORY VIA GITHUB_TOKEN
# ==============================================================================
import os
import sys
import subprocess

REPO_NAME = "upar_challenge_26_rws27"
REPO_DIR = os.path.join("/kaggle/working", REPO_NAME)

# Retrieve Kaggle Secret GITHUB_TOKEN
token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("GITHUB_TOKEN")
    print("[AUTH] Successfully retrieved GITHUB_TOKEN from Kaggle Secrets!")
except Exception as e:
    print(f"[AUTH] Note: Kaggle Secrets not accessible ({e}). Attempting public clone...")

if token:
    repo_url = f"https://{token}@github.com/Aditya-Lingam-9000/{REPO_NAME}.git"
else:
    repo_url = f"https://github.com/Aditya-Lingam-9000/{REPO_NAME}.git"

if not os.path.isdir(REPO_DIR):
    print(f"[GIT] Cloning repository into {REPO_DIR}...")
    res = subprocess.run(["git", "clone", repo_url, REPO_DIR], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[GIT ERROR] Clone failed:\n{res.stderr}")
        raise RuntimeError("Failed to clone private repository. Please verify GITHUB_TOKEN.")
    print(f"[GIT] Successfully cloned repository!")
else:
    print(f"[GIT] Repository already exists at {REPO_DIR}. Fetching latest commits from main...")
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    print(res.stdout)

# Add repository root to Python path and change working directory
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"[SYSTEM] Working directory set to: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 3: MASTER CHAMPIONSHIP CONFIGURATION PANEL
# ==============================================================================
import os
import sys
from sota_pipeline.config import SOTAConfig

# Modify any training hyperparameter here before running training.
EPOCHS = 10                  # Initial verification: 10 epochs. Set to 50-80 for full championship.
BATCH_SIZE_PER_GPU = 32      # 32 per GPU * 2 GPUs = Global batch size 64 (fits safely in 14.5GB VRAM)
ACCUMULATION_STEPS = 4       # Gradient accumulation: Effective global batch size = 256
BACKBONE_LR = 5e-5           # Conservative learning rate for pretrained visual backbone
ADAPTER_LR = 3.5e-4          # Higher learning rate for cross-modal heads and adapters
MIN_LR = 1e-6                # Minimum learning rate floor for Cosine Annealing
LOG_INTERVAL = 25            # Step interval for step-wise formatted console logs
EVAL_EVERY = 1               # Run full validation retrieval after EVERY epoch
VAL_QUERIES = 500            # Number of validation queries for retrieval benchmarking
MAX_SESSION_HOURS = 10.75    # Auto-save & graceful exit before Kaggle 12-hour session kill
CHECKPOINT_DIR = "checkpoints_sota"
OUTPUT_DIR = "sota_outputs"

# Auto-detect UPAR dataset root via dynamic deep scanner
DATASET_ROOT = None
try:
    DATASET_ROOT = SOTAConfig.resolve_dataset_root()
except FileNotFoundError as err:
    print(f"\n[DATASET RESOLUTION ERROR]\n{err}\n")
    raise err

print("=" * 80)
print(f"[CONFIG] Training Configuration:")
print(f"  - Epochs: {EPOCHS} (Initial Verification)")
print(f"  - Batch Size: {BATCH_SIZE_PER_GPU}/GPU (Global: {BATCH_SIZE_PER_GPU * 2}, Effective: {BATCH_SIZE_PER_GPU * 2 * ACCUMULATION_STEPS})")
print(f"  - Backbone LR: {BACKBONE_LR} | Adapters LR: {ADAPTER_LR} | Min LR: {MIN_LR}")
print(f"  - Eval Interval: Every {EVAL_EVERY} epoch(s) | Val Queries: {VAL_QUERIES}")
print(f"  - Dataset Root: {DATASET_ROOT}")
print(f"  - Checkpoint Dir: {os.path.abspath(CHECKPOINT_DIR)}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 4: HUGGINGFACE ACCELERATE DUAL T4 MULTI-GPU CONFIGURATION
# ==============================================================================
import os
import yaml

accelerate_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "MULTI_GPU",
    "num_machines": 1,
    "num_processes": 2,
    "gpu_ids": "all",
    "mixed_precision": "fp16",
    "main_process_port": 29500,
    "rdzv_backend": "static",
    "same_network": True,
    "ddp_config": {
        "find_unused_parameters": True,
    },
}

config_path = "accelerate_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(accelerate_config, f, default_flow_style=False)

print(f"[ACCELERATE] Generated multi-GPU configuration at: {os.path.abspath(config_path)}")


In [ ]:
# ==============================================================================
# CELL 5: LAUNCH SOTA CHAMPIONSHIP DISTRIBUTED TRAINING
# ==============================================================================
# Executes directly from the cloned repository (sota_pipeline.train).
# Features:
# 1. Baseline Pretrained Zero-Shot Validation (Epoch 0)
# 2. Lightning-fast RAM Preloading (2.5s from HDF5 container, zero disk bottlenecks)
# 3. Clean step-wise logs (NO tqdm bar clutter) with loss components, ETAs, speed, and VRAM
# 4. Full validation retrieval after EVERY epoch (mADM, CMC Rank-1/5/10, mAP)
# 5. Strictly saves ONLY best_model.pt and last_checkpoint.pt
# 6. Automatic resumption from last_checkpoint.pt if restarting session
import os
import sys
import shutil
import subprocess

# Auto-detect launcher: accelerate launch or torchrun fallback
if shutil.which("accelerate"):
    launcher = ["accelerate", "launch", "--config_file", "accelerate_config.yaml"]
else:
    launcher = ["torchrun", "--nproc_per_node=2"]

cmd = launcher + [
    "-m", "sota_pipeline.train",
    f"--epochs={EPOCHS}",
    f"--batch_size={BATCH_SIZE_PER_GPU}",
    f"--accumulation_steps={ACCUMULATION_STEPS}",
    f"--backbone_lr={BACKBONE_LR}",
    f"--adapter_lr={ADAPTER_LR}",
    f"--min_lr={MIN_LR}",
    f"--log_interval={LOG_INTERVAL}",
    f"--eval_every={EVAL_EVERY}",
    f"--val_queries={VAL_QUERIES}",
    f"--max_session_hours={MAX_SESSION_HOURS}",
    f"--checkpoint_dir={CHECKPOINT_DIR}",
    f"--output_dir={OUTPUT_DIR}",
    f"--dataset_root={DATASET_ROOT}",
    "--run_baseline",
]

print("[EXEC] Executing command:", " ".join(cmd))
print("=" * 80)
subprocess.run(cmd, check=True)


In [ ]:
# ==============================================================================
# CELL 6: POST-TRAINING INTEGRITY AUDIT & VERIFICATION
# ==============================================================================
# Executes directly from the cloned repository (sota_pipeline.verify).
# Verifies:
# 1. Existence and sizes of strictly best_model.pt and last_checkpoint.pt
# 2. Confirms zero intermediate checkpoint bloat
# 3. Verifies all 7 multi-session state components in each checkpoint
# 4. Validates model weight loading and executes a forward test pass
import os
import sys
import subprocess
import torch

audit_cmd = [
    sys.executable, "-m", "sota_pipeline.verify",
    f"--checkpoint_dir={CHECKPOINT_DIR}",
    "--device=cuda:0" if torch.cuda.is_available() else "--device=cpu",
]
print("[AUDIT] Running:", " ".join(audit_cmd))
subprocess.run(audit_cmd, check=True)
